# UABR — Phase 1: Environment, model tests, GPU smoke + resume gate
Works on **Colab (T4)** and **Kaggle (GPU T4 x2 / P100)**. Runs in ~15–20 min.

**Gate to pass before Phase 2:** all tests green · 40M model trains in FP16 without NaN · checkpoint resumes with continuous loss · throughput (tokens/s) recorded.


In [ ]:
import os, sys
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.exists("/kaggle")
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content"
    PERSIST = "/content/drive/MyDrive/uabr"
elif ON_KAGGLE:
    WORK = "/kaggle/working"
    PERSIST = "/kaggle/working/persist"
else:
    WORK, PERSIST = os.getcwd(), os.path.join(os.getcwd(), "persist")
os.makedirs(PERSIST, exist_ok=True)
print("colab" if ON_COLAB else "kaggle" if ON_KAGGLE else "local", "| WORK =", WORK, "| PERSIST =", PERSIST)

In [ ]:
REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"   # set "" to use uabr-slm.zip instead
import glob, subprocess
os.chdir(WORK)
if REPO_URL:
    REPO_DIR = f"{WORK}/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if os.path.exists(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)   # get latest pushed code
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    zips = glob.glob(f"{PERSIST}/uabr-slm.zip") + glob.glob("/kaggle/input/**/uabr-slm.zip", recursive=True)
    if not zips and ON_COLAB:
        from google.colab import files
        up = files.upload(); zips = [k for k in up if k.endswith(".zip")]
    assert zips, "uabr-slm.zip not found"
    subprocess.run(["unzip", "-oq", zips[0], "-d", WORK], check=True)
    REPO_DIR = f"{WORK}/uabr-slm"
os.chdir(REPO_DIR)
print(os.getcwd()); print(sorted(os.listdir()))
!git log --oneline -1

In [ ]:
!pip install -q pyyaml pytest
!python scripts/check_gpu.py

In [ ]:
!python -m pytest tests -q

## Synthetic data
The real tokenized corpus comes in Phase 2. Synthetic Markov-chain tokens let us test the *exact* 40M code path now (ideal loss ≈ 1.39).

In [ ]:
!python scripts/make_synthetic_data.py --out_dir data/processed/synthetic --n_train 5000000 --n_val 200000

## Smoke run: real 40M model, FP16, 300 steps (handbook §14)
Watch: loss falling, `grad_norm` finite, `tok_per_s`, `peak_mem_gb`, `data_frac` (fraction of time waiting on data — should be small).

In [ ]:
RUN = "runs/smoke_pretrain"
MIRROR = f"{PERSIST}/smoke_pretrain"
!rm -rf {RUN} {MIRROR}
!python src/train/pretrain.py --config configs/smoke_40m.yaml --max_steps 300 --save_every 100 --log_every 20 --mirror_dir {MIRROR}

## Restart-recovery proof
Resume **from the Drive/persistent copy** (as you would in a fresh session) and run 50 more steps. Loss at step 320+ must continue from ~step 300, not reset to ~9.4.

In [ ]:
!python src/train/pretrain.py --config configs/smoke_40m.yaml --resume {MIRROR} --output_dir {RUN} --max_steps 350 --log_every 10 --mirror_dir {MIRROR}

## 8. Gate report + full-pretraining time estimate

In [ ]:
import json, math
rows = [json.loads(l) for l in open(f"{RUN}/metrics.jsonl")]
tr = [r for r in rows if "loss" in r]; va = [r for r in rows if "val_loss" in r]
steady = [r["tok_per_s"] for r in tr if r["step"] > 40]   # skip warm-up / first-step overhead
tps = sorted(steady)[len(steady)//2]
before = [r for r in tr if r["step"] <= 300][-1]; after = [r for r in tr if r["step"] > 300][0]
hours = 1e9 / tps / 3600
print(f"median throughput   : {tps:,.0f} tokens/s")
print(f"peak GPU memory     : {max(r['peak_mem_gb'] for r in tr)} GB")
print(f"loss step 20 -> 300 : {tr[0]['loss']} -> {before['loss']}")
print(f"resume continuity   : step {before['step']} loss {before['loss']}  ->  step {after['step']} loss {after['loss']}")
print(f"val loss            : {[ (r['step'], r['val_loss']) for r in va]}")
print(f"\n1B-token pretraining estimate: {hours:.1f} GPU-hours  (x1.15 overhead -> {hours*1.15:.1f} h)")
print(f"= about {math.ceil(hours*1.15/11)} Kaggle session(s) of 11 h, or {math.ceil(hours*1.15/3)} Colab chunks of ~3 h")
checks = {
  "loss fell": before["loss"] < tr[0]["loss"] - 1.0,
  "no NaN": all(math.isfinite(r["loss"]) for r in tr),
  "resume continuous": abs(after["loss"] - before["loss"]) < 0.5,
}
for k, v in checks.items(): print(("PASS " if v else "FAIL ") + k)

## Speed test: eager vs `--compile`
Same 40M model, same data, 200 steps with `torch.compile`. The first ~60 steps include compilation (1–3 min), so they are skipped in the comparison. Uses a separate run folder, so the step 6–7 results are untouched.

In [ ]:
RUN_C = "runs/smoke_compile"
!rm -rf {RUN_C}
!python src/train/pretrain.py --config configs/smoke_40m.yaml --output_dir {RUN_C} --max_steps 200 --save_every 200 --log_every 20 --compile

In [ ]:
import json, math, statistics
def steady_tps(run, first_step, last_step):
    rows = [json.loads(l) for l in open(f"{run}/metrics.jsonl")]
    v = [r["tok_per_s"] for r in rows if "tok_per_s" in r and first_step < r["step"] <= last_step]
    return statistics.median(v) if v else float("nan")

def final_loss(run, step):
    rows = [json.loads(l) for l in open(f"{run}/metrics.jsonl") if '"loss"' in l]
    return [r["loss"] for r in rows if r["step"] == step][0]

eager = steady_tps(RUN, 40, 300)
comp = steady_tps(RUN_C, 60, 200)
speedup = comp / eager
print(f"eager    : {eager:,.0f} tokens/s")
print(f"compiled : {comp:,.0f} tokens/s   ({speedup:.2f}x)")
print(f"loss at step 200  eager {final_loss(RUN, 200)}  vs  compiled {final_loss(RUN_C, 200)}  (should be close)")
best = max(eager, comp)
h = 1e9 / best / 3600 * 1.15
print(f"\n1B-token estimate with the faster mode: {h:.1f} h  ->  {math.ceil(h/11)} Kaggle session(s) of 11 h")
if speedup > 1.10:
    print("RECOMMENDATION: use --compile for pretraining (and add it to every resume command).")
else:
    print("RECOMMENDATION: skip --compile (gain under 10% is not worth the compile time and risk).")